# Warstwa Gold — ekstrakcja macierzy cech PM2.5 (horyzont 24 h)

Milestone 3, zadanie 1: budowa i atomowa materializacja warstwy **Gold** (`data/gold/`) z `data/silver/pm25_pogoda_2023_2025.parquet` przez DuckDB oraz bezwyciekowy podział czasowy z buforem embargo. Kolejny notatnik (`09_model_pm25.ipynb`) wczyta zamrożony zbiór przez `gold.load_gold()`.

Logika znajduje się w `scripts/gold_features.py`.

## 1. Ekstrakcja macierzy cech (Feature Matrix)

1. **Próbkowanie `reservoir(500000 ROWS)` i uzasadnienie rozmiaru**: w instrukcji 100 000 wierszy było wartością przykładową. Zmierzone wyniki na `HistGradientBoostingRegressor` (`max_iter=100`) pokazują poprawę `MAE` do progu 500 tys. wierszy, po czym 256-koszykowy histogram się nasyca (`100k`: `MAE=5.492`, `R2=0.5777` | `300k`: `MAE=5.433`, `R2=0.5829` | **`500k`: `MAE=5.380`, `R2=0.5828`** | `1M`: `MAE=5.438`, `R2=0.5831`). Odpytujemy bezpośrednio `data/silver/pm25_pogoda_2023_2025.parquet` z `WHERE pm25 IS NOT NULL` w podzapytaniu, `SET threads=1` i `REPEATABLE (42)`.
2. **Brak wycieku danych (Data Leakage, horyzont 24 h)**: z `X` odrzucamy `pm25` (cel), `kod_stacji` i `data_czas_utc` (klucze), surowy kąt `wind_direction_10m` (zastąpiony przez `wiatr_u`, `wiatr_v`) oraz historię PM2.5 poniżej 24 h (`pm25_lag1..3`, `pm25_srednia3h..24h`, `pm25_max24h`). Zostaje **29 bezpiecznych cech** z `typ_stacji` i `typ_obszaru` jako `pl.Enum` oraz naturalnymi brakami `NULL` (bez imputacji i bez przeliczania lagów po próbkowaniu).
3. **Podział kwartalno-blokowy 80/20 + embargo `168 h`**: każdy z 12 kwartałów (`2023 Q1` … `2025 Q4`) dzielimy na 5 równych bloków czasowych (~18 dni) i losujemy (`seed=42`) 1 blok do zbioru testowego. Ze zbioru treningowego usuwamy (purging/embargo) wiersze w oknie **168 godzin (7 dni) bezpośrednio po każdym bloku testowym**, dzięki czemu żadna etykieta `y_test` nie przenika do cech `pm25_lag24` i `pm25_lag168` w `X_train`.
4. **Materializacja warstwy Gold (`data/gold/`)**: zapis atomowy `train.parquet`, `test.parquet` (kompresja Snappy, z zachowaniem `pl.Enum`) oraz `metadata.json` (SHA-256 źródła Silver, granice bloków testowych, liczebności, kontrakt schematu i wersje bibliotek). Kolejne kroki wczytują dane przez `gold.load_gold()`, który waliduje kontrakt przed treningiem.

In [1]:
from pathlib import Path
import sys

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import gold_features as gold
from scripts.pipeline_config import GOLD_DIR

gold.build_gold()
X_train, X_test, y_train, y_test = gold.load_gold()
train_df, test_df, meta = gold.load_gold(return_frames=True)

rows = meta["rows"]
print(f"Próbka DuckDB reservoir: {rows['sampled']:,} wierszy")
print(f"X_train: {X_train.shape} ({100 * rows['train_share']:.2f}%) | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape} ({100 * rows['test_share']:.2f}%) | y_test:  {y_test.shape}")
print(f"Wiersze odrzucone przez embargo ({meta['split']['embargo_hours']} h): {rows['embargo_purged']:,}")
for name in ("train.parquet", "test.parquet", "metadata.json"):
    p = GOLD_DIR / name
    print(f"  {p.relative_to(ROOT).as_posix()}: {p.stat().st_size / 1_000_000:.2f} MB")

Próbka DuckDB reservoir: 500,000 wierszy
X_train: (364415, 29) (78.42%) | y_train: (364415,)
X_test:  (100282, 29) (21.58%) | y_test:  (100282,)
Wiersze odrzucone przez embargo (168 h): 35,303
  data/gold/train.parquet: 21.29 MB
  data/gold/test.parquet: 5.86 MB
  data/gold/metadata.json: 0.01 MB


### Weryfikacja kontraktu Gold, braku wycieku lagów (`24 h` i `168 h`) i pokrycia pór roku

In [2]:
from datetime import timedelta

assert meta["rows"]["sampled"] == gold.SAMPLE_SIZE
assert set(X_train.columns) == set(gold.FEATURE_COLUMNS) and len(X_train.columns) == 29
assert set(X_train.columns).isdisjoint(gold.EXCLUDED_COLUMNS)
assert set(X_test.columns).isdisjoint(gold.EXCLUDED_COLUMNS)
assert X_train.schema["typ_stacji"] == gold.STATION_TYPE_ENUM
assert X_train.schema["typ_obszaru"] == gold.AREA_TYPE_ENUM
assert train_df["pm25"].null_count() == 0 and test_df["pm25"].null_count() == 0

train_ts = set(train_df["data_czas_utc"].to_list())
test_ts = set(test_df["data_czas_utc"].to_list())
assert len(train_ts & test_ts) == 0
assert len({t - timedelta(hours=24) for t in train_ts} & test_ts) == 0
assert len({t - timedelta(hours=168) for t in train_ts} & test_ts) == 0
assert train_df.select(pl.struct(pl.col("data_czas_utc").dt.year().alias("rok"), pl.col("data_czas_utc").dt.quarter().alias("kwartal")).n_unique()).item() == 12
assert test_df.select(pl.struct(pl.col("data_czas_utc").dt.year().alias("rok"), pl.col("data_czas_utc").dt.quarter().alias("kwartal")).n_unique()).item() == 12

train_by_q = train_df.group_by(pl.col("data_czas_utc").dt.quarter().alias("kwartal")).agg(
    pl.col("data_czas_utc").dt.year().n_unique().alias("lata_train"),
    pl.len().alias("wiersze_train"),
)
test_by_q = test_df.group_by(pl.col("data_czas_utc").dt.quarter().alias("kwartal")).agg(
    pl.col("data_czas_utc").dt.year().n_unique().alias("lata_test"),
    pl.len().alias("wiersze_test"),
)
(
    train_by_q.join(test_by_q, on="kwartal", how="inner")
    .with_columns(
        (100 * pl.col("wiersze_test") / (pl.col("wiersze_train") + pl.col("wiersze_test")))
        .round(2)
        .alias("udzial_test_proc")
    )
    .sort("kwartal")
)

kwartal,lata_train,wiersze_train,lata_test,wiersze_test,udzial_test_proc
i8,u32,u32,u32,u32,f64
1,3,88972,3,24824,21.81
2,3,94121,3,24938,20.95
3,3,87194,3,25529,22.65
4,3,94128,3,24991,20.98


In [3]:
display(
    X_train.null_count()
    .transpose(include_header=True, header_name="cecha", column_names=["braki_train"])
    .filter(pl.col("braki_train") > 0)
    .sort("braki_train", descending=True)
)
X_train.head(5)

cecha,braki_train
str,u32
"""pm25_lag168""",7004
"""pm25_lag24""",3751


temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wiatr_u,wiatr_v,cisnienie_zmiana3h,temperatura_zmiana24h,stopniodni_grzania,opad_suma6h,opad_suma24h,godzina,dzien_tygodnia,miesiac,weekend,swieto,dzien_wolny,godzina_sin,godzina_cos,miesiac_sin,miesiac_cos,typ_stacji,typ_obszaru,wysokosc_m,siatka_szerokosc,siatka_dlugosc,pm25_lag24,pm25_lag168
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,bool,bool,bool,f64,f64,f64,f64,enum,enum,f64,f64,f64,f64,f64
-0.7,87.0,0.0,1008.5,19.9,19.389964,-4.476526,-1.7,2.0,15.7,-1.4433e-15,0.2,14,5,1,false,false,false,-0.5,-0.866025,0.0,1.0,"""tło""","""miejski""",152.0,51.28295,15.891891,null,null
-1.1,92.0,0.0,1007.7,16.2,16.177798,-0.847842,-1.1,2.2,16.1,-1.4433e-15,0.2,16,5,1,false,false,false,-0.866025,-0.5,0.0,1.0,"""tło""","""miejski""",152.0,51.28295,15.891891,null,null
-1.0,94.0,0.0,1006.5,18.4,18.4,3.3800e-15,-1.1,2.1,16.0,-1.4433e-15,0.2,20,5,1,false,false,false,-0.866025,0.5,0.0,1.0,"""tło""","""miejski""",152.0,51.28295,15.891891,null,null
0.3,89.0,0.0,1002.4,22.4,22.27729,-2.341438,-1.2,2.6,14.7,-1.4433e-15,-1.3323e-15,4,6,1,true,false,true,0.866025,0.5,0.0,1.0,"""tło""","""miejski""",152.0,51.28295,15.891891,null,null
0.3,90.0,0.0,1001.2,20.5,20.496878,-0.357774,-1.2,2.7,14.7,-1.4433e-15,-1.3323e-15,7,6,1,true,false,true,0.965926,-0.258819,0.0,1.0,"""tło""","""miejski""",152.0,51.28295,15.891891,null,null
